In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import shutil
import torch.nn as nn
import os
import pandas as pd
import random
import torch.nn.functional as F

from PatchDataset import load_dataset
from torch.utils.data import DataLoader,Subset,Dataset
from torchvision import datasets, transforms
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix
from transformers import CLIPProcessor, CLIPModel



In [2]:
PATCH_SIZE = 224
BATCH_SIZE = 32
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PATCHES_ROOT = "patches_dataset/patches_v1_seed42"
csv_path = os.path.join(PATCHES_ROOT, "metadata.csv")


In [3]:
# loading the dataset
df = pd.read_csv(csv_path)
df["patch_id"] = df["patch_id"].astype(int)
df["label"] = df["label"].astype(int)
df["type"] = df["type"].astype(str)

train_dataset,test_dataset,train_loader,test_loader = load_dataset(df,PATCHES_ROOT,BATCH_SIZE)

for xb,yb in train_loader:
    print(xb.shape,yb.shape,min(yb),max(yb))
    break

train/test: 7056 1764
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0.) tensor(1.)


In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score, average_precision_score

# Load pretrained ViT for binary classification
model_vit = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=2,  # binary: tree/no-tree
    ignore_mismatched_sizes=True  # replace classification head
).to(device)

processor_vit = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [ ]:

# Freeze backbone, train only head (fast start)
for param in model_vit.vit.parameters():
    param.requires_grad = False

# Unfreeze gradually for better results
unfrozen_layers = ['vit.encoder.layer.11', 'classifier']
for name, param in model_vit.named_parameters():
    if any(layer in name for layer in unfrozen_layers):
        param.requires_grad = True

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model_vit.parameters()), lr=2e-5)
criterion = nn.CrossEntropyLoss()
scheduler = CosineAnnealingLR(optimizer, T_max=10)

def train_vit_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device).long()
        
        # ViT expects processed inputs
        inputs = processor_vit(images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        optimizer.zero_grad()
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), correct / total

def eval_vit(model, loader, device):
    model.eval()
    all_scores, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(loader):
            images, labels = images.to(device), labels.to(device).long()
            inputs = processor_vit(images, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)
            tree_scores = probs[:, 1].cpu().numpy()  # P(tree|patch)
            
            all_scores.extend(tree_scores)
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_scores), np.array(all_labels)

In [7]:

# Train 10 epochs
print("Training ViT...")
best_auc = 0
for epoch in range(10):
    train_loss, train_acc = train_vit_epoch(model_vit, train_loader, optimizer, criterion, device)
    scheduler.step()
    
    scores, labels = eval_vit(model_vit, test_loader, device)
    auc = roc_auc_score(labels, scores)
    ap = average_precision_score(labels, scores)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Acc={train_acc:.3f}, "
          f"Test AUC={auc:.4f}, AP={ap:.4f}")
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model_vit.state_dict(), 'best_vit_tree_classifier.pth')

print(f"Best AUC: {best_auc:.4f}")


Training ViT...


  0%|          | 0/56 [00:08<?, ?it/s]


NameError: name 'F' is not defined

In [ ]:

# Your existing ROC/PR plots work directly:
fpr, tpr, _ = roc_curve(labels, scores)
plt.plot(fpr, tpr, label=f"ViT AUC={auc:.3f}")
plt.legend()
plt.show()
